# 03a - CPU-intensive Experiments (A/B)

## Overview
This notebook runs CPU-intensive experiments using **LightGBM** for carbon emission prediction.

### Experiment Types
| Code | Experiment Name | Description | Model |
|------|----------------|-------------|-------|
| **A** | Country Self-Modeling | Train and test using single-country data | LightGBM |
| **B** | Global Baseline | Train on global data (excluding target), test on target country | LightGBM |
| **B_dev** | Developed Countries Baseline | Train on developed countries data, test on target country | LightGBM |

### Hardware Requirements
- **CPU**: Multi-core recommended (LightGBM is CPU-optimized)
- **Memory**: 16GB+ recommended
- **GPU**: Not required

In [1]:
# Global Country Loop Experiments - Using Global Experiment Framework
import pandas as pd
import numpy as np
import os
import time
import sys
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

print("✅ Sklearn feature name warnings suppressed")

# Resolve base directory for Transfer project
BASE_DIR = os.environ.get("TRANSFER_DIR")
if not BASE_DIR:
    cwd = os.getcwd()
    if os.path.basename(cwd) == "Transfer":
        BASE_DIR = cwd
    elif os.path.isdir(os.path.join(cwd, "Transfer")):
        BASE_DIR = os.path.join(cwd, "Transfer")
    else:
        BASE_DIR = "/root/autodl-fs/Transfer"
BASE_DIR = os.path.abspath(BASE_DIR)
if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)
print(f"✅ BASE_DIR set to: {BASE_DIR}")

# Import Global Experiment Runner
try:
    from model_trainer_global import run_global_experiments, GlobalExperimentRunner
    print("✅ Successfully imported Global Experiment Runner")
except ImportError:
    sys.path.append(BASE_DIR)
    try:
        from model_trainer_global import run_global_experiments, GlobalExperimentRunner
        print("✅ Imported Global Experiment Runner via BASE_DIR")
    except ImportError:
        sys.path.append(r'd:\9_Projects\Transfer learning\python modules')
        from model_trainer_global import run_global_experiments, GlobalExperimentRunner
        print("✅ Imported Global Experiment Runner via absolute path")

# --- Configuration ---
data_dir = os.path.join(BASE_DIR, 'data')
processed_hdf5_path = os.path.join(data_dir, 'processed_feature_data.h5')
results_dir = os.path.join(BASE_DIR, 'results')
os.makedirs(results_dir, exist_ok=True)

# Verify file paths
print(f"Current working directory: {os.getcwd()}")
print(f"Data file path: {os.path.abspath(processed_hdf5_path)}")
print(f"File exists: {os.path.exists(processed_hdf5_path)}")

# Target column definition
TARGET_COLS = [
    'log_Scope1', 'log_Scope2', 'log_Scope3_upstream', 'log_Scope3_downLA', 
    'log_Scope3_prod', 'log_Scope_total',
]

# --- Load Global Data ---
print(f"Loading global feature-engineered data from {processed_hdf5_path}...")
try:
    df = pd.read_hdf(processed_hdf5_path, key='processed_features')
    
    # Global data quality check
    print("🔍 Global Data Quality Check:")
    
    # Ensure target columns exist
    for col in TARGET_COLS:
        if col not in df.columns:
            raise ValueError(f"Target column {col} not found in loaded data!")

    # Check geographic location column (must have 'loc' column for country identification)
    if 'loc' not in df.columns:
        geo_columns = [col for col in df.columns if col.lower() in ['country', 'location']]
        if geo_columns:
            df = df.rename(columns={geo_columns[0]: 'loc'})
            print(f"  - Renamed {geo_columns[0]} to 'loc'")
        else:
            raise ValueError("Missing country/region identifier column in data!")
    
    # Enhanced loc column cleaning
    print(f"  - Found {df['loc'].nunique()} unique loc values in raw data")
    
    # Check for various anomaly types
    null_mask = df['loc'].isnull()
    empty_str_mask = df['loc'] == ''
    whitespace_mask = df['loc'].astype(str).str.strip() == ''
    short_mask = df['loc'].astype(str).str.len() < 2
    
    print(f"  - NULL values: {null_mask.sum():,}")
    print(f"  - Empty strings: {empty_str_mask.sum():,}")
    print(f"  - Whitespace-only strings: {whitespace_mask.sum():,}")
    print(f"  - Strings with length < 2: {short_mask.sum():,}")
    
    # Strict data cleaning
    initial_count = len(df)
    
    valid_mask = (
        df['loc'].notna() &
        (df['loc'] != '') &
        (df['loc'].astype(str).str.strip() != '') &
        (df['loc'].astype(str).str.len() >= 2) &
        (df['loc'].astype(str).str.isalpha()) &
        (df['loc'].astype(str).str.len() <= 10)
    )
    
    df = df[valid_mask].copy()
    removed_count = initial_count - len(df)
    
    print(f"  - Removed samples with anomalous loc values: {removed_count:,}")
    print(f"  - Found {df['loc'].nunique()} valid countries/regions after cleaning")

    # Get feature column names
    exclude_cols = TARGET_COLS + ['gvkey', 'fiscalyear', 'loc']
    other_exclude = [col for col in df.columns if 'sector' in col.lower() or 'gics' in col.lower()]
    exclude_cols.extend(other_exclude)
    
    FEATURE_COLS = [col for col in df.columns if col not in exclude_cols]
    
    # Check feature column data types and quality
    print(f"  - Number of feature columns: {len(FEATURE_COLS)}")
    feature_nans = df[FEATURE_COLS].isnull().sum().sum()
    print(f"  - NaN count in feature columns: {feature_nans}")
    
    # Check target column NaN status
    for col in TARGET_COLS:
        nan_count = df[col].isnull().sum()
        valid_count = df[col].notnull().sum()
        print(f"  - {col}: {valid_count:,} valid, {nan_count:,} missing")
    
    # Remove rows where all targets are NaN
    initial_count_targets = len(df)
    df = df.dropna(subset=TARGET_COLS, how='all')
    removed_count_targets = initial_count_targets - len(df)
    
    print(f"  - Removed samples with all-NaN targets: {removed_count_targets:,}")
    print(f"  - Retained samples: {len(df):,}")
    
    print("✅ Global data loaded successfully!")
    print(f"Final data shape: {df.shape}")
    print(f"Final feature count: {len(FEATURE_COLS)}")
    print(f"Valid country count: {df['loc'].nunique()}")
    
    # Show top 15 countries by sample size
    country_counts = df['loc'].value_counts().head(15)
    print(f"\n📊 Top 15 Countries by Sample Size:")
    for i, (country, count) in enumerate(country_counts.items(), 1):
        print(f"  {i:2d}. '{country}': {count:,}")
    
except Exception as e:
    print(f"❌ Failed to load data: {e}")
    if os.path.exists(data_dir):
        print(f"data directory exists, contains files: {os.listdir(data_dir)}")
    else:
        print(f"data directory does not exist: {data_dir}")
    raise

✅ Sklearn feature name warnings suppressed
✅ BASE_DIR set to: /autodl-fs/data/Transfer


✅ Successfully imported Global Experiment Runner
Current working directory: /autodl-fs/data/Transfer
Data file path: /autodl-fs/data/Transfer/data/processed_feature_data.h5
File exists: True
Loading global feature-engineered data from /autodl-fs/data/Transfer/data/processed_feature_data.h5...


🔍 Global Data Quality Check:
  - Found 149 unique loc values in raw data


  - NULL values: 0
  - Empty strings: 24,271
  - Whitespace-only strings: 24,271
  - Strings with length < 2: 24,271


  - Removed samples with anomalous loc values: 24,271


  - Found 148 valid countries/regions after cleaning
  - Number of feature columns: 100


  - NaN count in feature columns: 0
  - log_Scope1: 199,290 valid, 1,274,180 missing
  - log_Scope2: 199,290 valid, 1,274,180 missing
  - log_Scope3_upstream: 199,290 valid, 1,274,180 missing
  - log_Scope3_downLA: 85,223 valid, 1,388,247 missing
  - log_Scope3_prod: 85,223 valid, 1,388,247 missing
  - log_Scope_total: 199,290 valid, 1,274,180 missing


  - Removed samples with all-NaN targets: 1,274,180
  - Retained samples: 199,290
✅ Global data loaded successfully!
Final data shape: (199290, 110)
Final feature count: 100
Valid country count: 117

📊 Top 15 Countries by Sample Size:
   1. 'USA': 38,379
   2. 'JPN': 23,682
   3. 'CHN': 23,057
   4. 'GBR': 10,881
   5. 'KOR': 10,737
   6. 'TWN': 8,959
   7. 'IND': 7,739
   8. 'AUS': 6,644
   9. 'HKG': 6,089
  10. 'CAN': 5,369
  11. 'FRA': 4,011
  12. 'DEU': 3,676
  13. 'SWE': 3,163
  14. 'CHE': 3,051
  15. 'MYS': 2,691


In [2]:
# --- Global Country Loop Experiment Configuration (Multi-mode) ---

import time
import sys
import os

print("🔧 Adding python modules directory to path...")
sys.path.append(os.path.abspath('../Transfer'))
print("✅ Path added")

try:
    from model_trainer_global import GlobalExperimentRunner, run_global_experiments
    print("✅ Successfully imported GlobalExperimentRunner and run_global_experiments")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("Current sys.path:")
    for path in sys.path:
        print(f"  - {path}")

# 🎯 Experiment Configuration Parameters
EXPERIMENT_CONFIG = {
    # Experiment parameters
    'min_samples': 50,                      # Minimum sample size threshold
    'experiments': ['A', 'B', 'B_dev'],     # Experiment types to run (CPU-intensive only)
    'random_state': 42,                     # Random seed
    'force_cpu': True,                      # Force CPU device for runner display and PyTorch components
    
    # LGBM hyperparameter optimization (for Experiments A/B/B_dev)
    'n_trials_lgbm': 20,                    # LGBM hyperparameter trial count
}

# Available experiment types
ALL_EXPERIMENTS = {
    'A': 'Country Self-Modeling - LightGBM single-country training and testing',
    'B': 'Global Baseline - LightGBM global training → single-country testing',
    'B_dev': 'Developed Countries Baseline - LightGBM developed countries training → single-country testing',
}

print("🌍 Global Country Loop Experiment Configuration:")
print("=" * 70)
print(f"Selected experiment types: {EXPERIMENT_CONFIG['experiments']}")
print(f"\n📋 Available Experiment Types (CPU-intensive):")
for exp_code, exp_desc in ALL_EXPERIMENTS.items():
    selected = "✓" if exp_code in EXPERIMENT_CONFIG['experiments'] else " "
    print(f"  [{selected}] {exp_code}: {exp_desc}")
print(f"\nMinimum sample threshold: {EXPERIMENT_CONFIG['min_samples']}")
print(f"LGBM hyperparameter trials: {EXPERIMENT_CONFIG['n_trials_lgbm']}")
print("=" * 70)

# 🎯 Run Mode Selection
print(f"\n🎯 Available Run Modes:")
print("1. debug_mode    - Debug mode: Run on 3 major countries (USA, JPN, CHN)")
print("2. full_auto     - Full auto mode: Run on all countries meeting threshold")
print("3. custom_list   - Custom mode: Run on specified country list")

# 🔧 Select run mode here
RUN_MODE = "full_auto"  # Options: "debug_mode", "full_auto", "custom_list"

# Configure target countries and parameters based on run mode
if RUN_MODE == "debug_mode":
    print(f"\n🎯 Run Mode: {RUN_MODE} (Debug mode)")
    target_countries = ['USA', 'JPN', 'CHN']
    EXPERIMENT_CONFIG['n_trials_lgbm'] = 10
    print("🔧 Debug mode parameter adjustments:")
    print(f"  - LGBM trial count: {EXPERIMENT_CONFIG['n_trials_lgbm']} (reduced)")
    
elif RUN_MODE == "full_auto":
    print(f"\n🎯 Run Mode: {RUN_MODE} (Full auto mode)")
    target_countries = "auto"
    print("🌍 Will automatically select all countries meeting sample threshold")
    print("📊 Using full parameters for experiments")
    
elif RUN_MODE == "custom_list":
    print(f"\n🎯 Run Mode: {RUN_MODE} (Custom mode)")
    target_countries = [
        'USA', 'JPN', 'CHN', 'GBR', 'KOR', 'TWN', 'IND', 'AUS'
    ]
    print(f"🎯 Custom country list: {target_countries}")
    print("📊 Using full parameters for experiments")
    
else:
    raise ValueError(f"Unsupported run mode: {RUN_MODE}")

print(f"Target countries: {target_countries}")

# --- Data Validation ---
print(f"\n{'='*60}")
print(f"📊 Pre-run Data Validation...")
print(f"{'='*60}")

print(f"📊 Global Data Statistics:")
print(f"  - Total samples: {len(df):,}")
print(f"  - Total countries: {df['loc'].nunique()}")
print(f"  - Feature count: {len(FEATURE_COLS)}")
print(f"  - Target count: {len(TARGET_COLS)}")

# Data validation based on run mode
if RUN_MODE == "full_auto":
    print(f"\n🌍 Full auto mode: Filtering eligible countries...")
    print(f"Filter criteria: sample_count >= {EXPERIMENT_CONFIG['min_samples']}")
    
    valid_countries = []
    country_stats = []
    
    for country in df['loc'].unique():
        country_data = df[df['loc'] == country]
        country_samples = len(country_data)
        
        all_targets_valid = country_data[TARGET_COLS].notnull().all(axis=1)
        feature_complete = country_data[FEATURE_COLS].notnull().all(axis=1)
        usable_mask = all_targets_valid & feature_complete
        usable_samples = usable_mask.sum()
        
        country_stats.append({
            'country': country,
            'total_samples': country_samples,
            'usable_samples': usable_samples,
            'usable_rate': usable_samples / country_samples * 100 if country_samples > 0 else 0
        })
        
        if usable_samples >= EXPERIMENT_CONFIG['min_samples']:
            valid_countries.append(country)
    
    country_stats.sort(key=lambda x: x['usable_samples'], reverse=True)
    
    print(f"\n📊 Country Data Quality Statistics (sorted by usable samples):")
    print(f"{'Country':<8} {'Total':<12} {'Usable':<12} {'Rate':<10} {'Status':<10}")
    print("-" * 55)
    
    for i, stats in enumerate(country_stats):
        country = stats['country']
        total = stats['total_samples']
        usable = stats['usable_samples']
        rate = stats['usable_rate']
        status = "✅ Pass" if usable >= EXPERIMENT_CONFIG['min_samples'] else "❌ Fail"
        
        print(f"{country:<8} {total:<12,} {usable:<12,} {rate:<9.1f}% {status}")
        
        if i >= 19:
            remaining = len(country_stats) - 20
            if remaining > 0:
                print(f"... {remaining} more countries not shown")
            break
    
    target_countries = valid_countries
    print(f"\n✅ Auto-selected {len(target_countries)} eligible countries")
    print(f"First 10 countries: {target_countries[:10]}")
    
else:
    print(f"\n📋 Target Country Data Validation:")
    valid_target_countries = []
    
    for country in target_countries:
        if country in df['loc'].values:
            country_data = df[df['loc'] == country]
            country_samples = len(country_data)
            
            all_targets_valid = country_data[TARGET_COLS].notnull().all(axis=1)
            feature_complete = country_data[FEATURE_COLS].notnull().all(axis=1)
            usable_mask = all_targets_valid & feature_complete
            usable_samples = usable_mask.sum()
            
            print(f"  - {country}: {country_samples:,} total, {usable_samples:,} usable ({usable_samples/country_samples*100:.1f}%)")
            
            if usable_samples >= EXPERIMENT_CONFIG['min_samples']:
                valid_target_countries.append(country)
                print(f"    ✅ Meets minimum sample requirement")
            else:
                print(f"    ❌ Does not meet minimum sample requirement (need {EXPERIMENT_CONFIG['min_samples']})")
        else:
            print(f"  - {country}: ❌ Not in data")
    
    target_countries = valid_target_countries

print(f"\n✅ Final valid target country count: {len(target_countries)}")
if len(target_countries) <= 10:
    print(f"✅ Final valid target countries: {target_countries}")
else:
    print(f"✅ First 10 countries: {target_countries[:10]}")
    print(f"   Total {len(target_countries)} countries...")

# Estimate experiment time
total_experiments = len(target_countries) * len(EXPERIMENT_CONFIG['experiments'])
estimated_time_per_exp = 2 if RUN_MODE == "debug_mode" else 5
estimated_total_time = total_experiments * estimated_time_per_exp

print(f"\n⏱️ Experiment Scale Estimate:")
print(f"  - Target countries: {len(target_countries)}")
print(f"  - Experiment types: {len(EXPERIMENT_CONFIG['experiments'])}")
print(f"  - Total experiments: {total_experiments}")
print(f"  - Estimated total time: {estimated_total_time:.0f} minutes ({estimated_total_time/60:.1f} hours)")

print(f"\n✅ Configuration and validation phase complete!")

🔧 Adding python modules directory to path...
✅ Path added
✅ Successfully imported GlobalExperimentRunner and run_global_experiments
🌍 Global Country Loop Experiment Configuration:
Selected experiment types: ['A', 'B', 'B_dev']

📋 Available Experiment Types (CPU-intensive):
  [✓] A: Country Self-Modeling - LightGBM single-country training and testing
  [✓] B: Global Baseline - LightGBM global training → single-country testing
  [✓] B_dev: Developed Countries Baseline - LightGBM developed countries training → single-country testing

Minimum sample threshold: 50
LGBM hyperparameter trials: 20

🎯 Available Run Modes:
1. debug_mode    - Debug mode: Run on 3 major countries (USA, JPN, CHN)
2. full_auto     - Full auto mode: Run on all countries meeting threshold
3. custom_list   - Custom mode: Run on specified country list

🎯 Run Mode: full_auto (Full auto mode)
🌍 Will automatically select all countries meeting sample threshold
📊 Using full parameters for experiments
Target countries: auto




📊 Country Data Quality Statistics (sorted by usable samples):
Country  Total        Usable       Rate       Status    
-------------------------------------------------------
USA      38,379       15,514       40.4     % ✅ Pass
CHN      23,057       13,988       60.7     % ✅ Pass
JPN      23,682       11,099       46.9     % ✅ Pass
KOR      10,737       5,476        51.0     % ✅ Pass
TWN      8,959        4,315        48.2     % ✅ Pass
IND      7,739        3,618        46.8     % ✅ Pass
HKG      6,089        2,678        44.0     % ✅ Pass
GBR      10,881       2,326        21.4     % ✅ Pass
AUS      6,644        2,220        33.4     % ✅ Pass
CAN      5,369        2,115        39.4     % ✅ Pass
FRA      4,011        1,397        34.8     % ✅ Pass
SWE      3,163        1,355        42.8     % ✅ Pass
DEU      3,676        1,349        36.7     % ✅ Pass
MYS      2,691        1,177        43.7     % ✅ Pass
THA      2,453        1,155        47.1     % ✅ Pass
CHE      3,051        1,031  

In [3]:
# --- Experiment Execution Cell ---

# Configure Optuna logging
import optuna
import logging
import pickle
from datetime import datetime

# Clean and reconfigure Optuna logging to avoid duplicate output
optuna_logger = optuna.logging.get_logger("optuna")
for handler in optuna_logger.handlers[:]:
    optuna_logger.removeHandler(handler)

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("🔧 Log configuration optimized, debug mode enabled")

# --- Helper: build results dataframe (CPU experiments) ---
def build_cpu_results_df(experiment_results, target_cols):
    results_data = []
    for result in experiment_results:
        country = result.get('country', 'Unknown')
        exp_type_raw = result.get('experiment_type', 'Unknown')
        
        # Standardize experiment type names
        if 'Country_Self_Modeling' in exp_type_raw:
            exp_type = 'A'
        elif 'Global_Baseline' in exp_type_raw:
            exp_type = 'B'
        elif 'Developed_Baseline' in exp_type_raw:
            exp_type = 'B_dev'
        else:
            exp_type = exp_type_raw
        
        r2_scores = {}
        r2_avg = result.get('R2_Average', np.nan)
        for target in target_cols:
            r2_key = f'R2_{target}'
            r2_scores[target] = result.get(r2_key, np.nan)
        
        mae_values = [result.get(f'MAE_{target}', np.nan) for target in target_cols]
        rmse_values = [result.get(f'RMSE_{target}', np.nan) for target in target_cols]
        mae_avg = np.nanmean(mae_values) if mae_values else np.nan
        rmse_avg = np.nanmean(rmse_values) if rmse_values else np.nan
        
        training_samples = result.get('training_samples', 0)
        valid_training_samples = result.get('valid_training_samples', 0)
        data_efficiency = result.get('data_efficiency', 0)
        
        results_data.append({
            'country': country,
            'experiment_type': exp_type,
            'experiment_type_full': exp_type_raw,
            'R2_Average': r2_avg,
            'MAE_Average': mae_avg,
            'RMSE_Average': rmse_avg,
            'training_samples': training_samples,
            'valid_training_samples': valid_training_samples,
            'data_efficiency': data_efficiency,
            **r2_scores
        })
    
    return pd.DataFrame(results_data)

# --- Helper: save results to Excel (English format) ---
def save_results_excel(results_df, results_dir, prefix, metadata):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_path = os.path.join(results_dir, f"{prefix}_{timestamp}.xlsx")
    
    summary_by_exp = results_df.groupby('experiment_type').agg({
        'R2_Average': ['count', 'mean', 'std', 'min', 'max'],
        'MAE_Average': 'mean',
        'RMSE_Average': 'mean'
    }).round(4)
    
    summary_by_country = results_df.groupby('country').agg({
        'R2_Average': ['count', 'mean', 'std']
    }).round(4)
    
    metadata_df = pd.DataFrame(list(metadata.items()), columns=['Field', 'Value'])
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='Results', index=False)
        summary_by_exp.to_excel(writer, sheet_name='Summary_By_Experiment')
        summary_by_country.to_excel(writer, sheet_name='Summary_By_Country')
        metadata_df.to_excel(writer, sheet_name='Metadata', index=False)
    
    return excel_path

# --- Checkpoint Setup ---
checkpoint_path = os.path.join(results_dir, "cpu_ab_checkpoint.pkl")
checkpoint_results = []
completed_countries = set()

# Set to True to ignore existing checkpoint and rerun all countries
RESET_CHECKPOINT = False
if RESET_CHECKPOINT and os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print("🧹 Checkpoint cleared. Will rerun all countries.")

if os.path.exists(checkpoint_path):
    try:
        with open(checkpoint_path, "rb") as f:
            checkpoint_data = pickle.load(f)
        checkpoint_results = checkpoint_data.get("results", [])
        completed_countries = set(checkpoint_data.get("completed_countries", []))
        print(f"✅ Loaded checkpoint: {len(completed_countries)} countries, {len(checkpoint_results)} results")
    except Exception as e:
        print(f"⚠️ Failed to load checkpoint, starting fresh: {e}")

# --- Preview Experiment Plan ---
if target_countries:
    print(f"\n{'='*60}")
    print(f"📋 Experiment Plan Preview...")
    print(f"{'='*60}")
    
    try:
        preview_runner = GlobalExperimentRunner(
            df, FEATURE_COLS, TARGET_COLS, EXPERIMENT_CONFIG['random_state'],
            force_cpu=EXPERIMENT_CONFIG.get('force_cpu', False)
        )
        
        print(f"✅ GlobalExperimentRunner initialized successfully")
        print(f"📊 Expected total experiments: {len(target_countries) * len(EXPERIMENT_CONFIG['experiments'])}")
        print(f"✅ Eligible target countries: {len(target_countries)}")
        
        print(f"\n📋 Target Country Details:")
        for i, country in enumerate(target_countries, 1):
            if country in preview_runner.country_quality:
                country_info = preview_runner.country_quality[country]
                print(f"  {i:2d}. {country}: {country_info['usable']:,} samples ({country_info['usable_rate']:.1f}% valid)")
            else:
                print(f"  {i:2d}. {country}: ⚠️ Quality info missing")
        
        print(f"\n🚀 Ready to start experiments...")
        
    except Exception as e:
        print(f"❌ GlobalExperimentRunner initialization failed: {e}")
        import traceback
        traceback.print_exc()
        preview_runner = None

    # --- Start Experiment Execution ---
    if preview_runner:
        print(f"\n{'='*60}")
        print(f"🚀 Starting CPU-intensive experiments (A/B)...")
        print(f"{'='*60}")
        
        start_time = time.time()
        all_results = list(checkpoint_results)
        remaining_countries = [c for c in target_countries if c not in completed_countries]
        print(f"✅ Remaining countries to run: {len(remaining_countries)}")
        
        if not remaining_countries:
            print("✅ All target countries already completed. Using checkpoint results.")
        
        try:
            for idx, country in enumerate(remaining_countries, 1):
                print(f"\n▶️ Running {idx}/{len(remaining_countries)}: {country}")
                country_results = run_global_experiments(
                    df=df,
                    features=FEATURE_COLS,
                    targets=TARGET_COLS,
                    countries=[country],
                    **EXPERIMENT_CONFIG
                )
                all_results.extend(country_results)
                completed_countries.add(country)
                
                # Save checkpoint after each country
                checkpoint_data = {
                    "completed_countries": sorted(list(completed_countries)),
                    "results": all_results
                }
                with open(checkpoint_path, "wb") as f:
                    pickle.dump(checkpoint_data, f)
                print(f"✅ Checkpoint saved: {len(completed_countries)} countries, {len(all_results)} results")
            
            end_time = time.time()
            total_time = (end_time - start_time) / 60
            
            print(f"\n🎉 CPU experiments complete!")
            print(f"Total time: {total_time:.2f} minutes")
            print(f"Completed experiments: {len(all_results)}")
            
            experiment_results = all_results
            experiment_duration = total_time
            global_results = all_results
            
            # Immediate Excel export
            if experiment_results:
                results_df = build_cpu_results_df(experiment_results, TARGET_COLS)
                metadata = {
                    "Run_Mode": RUN_MODE,
                    "Experiments": ", ".join(EXPERIMENT_CONFIG['experiments']),
                    "Min_Samples": EXPERIMENT_CONFIG['min_samples'],
                    "LGBM_Trials": EXPERIMENT_CONFIG['n_trials_lgbm'],
                    "Target_Countries": len(target_countries),
                    "Start_Time": datetime.fromtimestamp(start_time).strftime("%Y-%m-%d %H:%M:%S"),
                    "End_Time": datetime.fromtimestamp(end_time).strftime("%Y-%m-%d %H:%M:%S"),
                    "Total_Time_Min": round(total_time, 2)
                }
                excel_path = save_results_excel(
                    results_df,
                    results_dir,
                    prefix="Global_Carbon_Emissions_Experiment_Results_CPU_AB",
                    metadata=metadata
                )
                print(f"💾 Results saved immediately to: {excel_path}")
            
            print(f"\n📊 Experiment Results Overview:")
            print(f"  - Successful experiments: {len([r for r in all_results if r.get('status') == 'success' or 'status' not in r])}")
            print(f"  - Failed experiments: {len([r for r in all_results if r.get('status') == 'failed'])}")
            
            # Group by experiment type
            exp_types = {}
            for result in all_results:
                exp_type = result.get('experiment_type', 'Unknown')
                if exp_type not in exp_types:
                    exp_types[exp_type] = {'success': 0, 'failed': 0}
                
                status = result.get('status', 'success')
                if status != 'failed' and not np.isnan(result.get('R2_Average', np.nan)):
                    exp_types[exp_type]['success'] += 1
                else:
                    exp_types[exp_type]['failed'] += 1
            
            print(f"\n📈 Statistics by Experiment Type:")
            for exp_type, counts in exp_types.items():
                total = counts['success'] + counts['failed']
                success_rate = counts['success'] / total * 100 if total > 0 else 0
                print(f"  - {exp_type}: {counts['success']}/{total} success ({success_rate:.1f}%)")
            
        except Exception as e:
            print(f"❌ Error during experiment: {e}")
            import traceback
            traceback.print_exc()
            global_results = []
            experiment_results = []
            experiment_duration = 0
    else:
        print(f"\n⚠️ Skipping experiment run:")
        print(f"  - GlobalExperimentRunner initialization failed")
        global_results = []
        experiment_results = []
        experiment_duration = 0
else:
    print(f"\n⚠️ Skipping experiment run:")
    print(f"  - No valid target countries")
    global_results = []
    experiment_results = []
    experiment_duration = 0

print(f"\n✅ Experiment execution phase complete!")
if experiment_results:
    print(f"✅ Experiment data ready for analysis")
    print(f"📊 Variable 'experiment_results' contains {len(experiment_results)} experiment results")
    print(f"⏱️ Variable 'experiment_duration' records total time: {experiment_duration:.2f} minutes")
else:
    print(f"⚠️ No experiment results, please check data and configuration")

🔧 Log configuration optimized, debug mode enabled
✅ Loaded checkpoint: 14 countries, 42 results

📋 Experiment Plan Preview...


🌍 Global Experiment Runner initialized. Computing device: CPU
📊 Found 117 countries/regions
📊 Developed countries count: 35
🔍 Analyzing global data quality...


📈 Top 10 Countries by Sample Count:
   1. USA: 38,379 usable samples (100.0%)
   2. JPN: 23,682 usable samples (100.0%)
   3. CHN: 23,057 usable samples (100.0%)
   4. GBR: 10,881 usable samples (100.0%)
   5. KOR: 10,737 usable samples (100.0%)
   6. TWN: 8,959 usable samples (100.0%)
   7. IND: 7,739 usable samples (100.0%)
   8. AUS: 6,644 usable samples (100.0%)
   9. HKG: 6,089 usable samples (100.0%)
  10. CAN: 5,369 usable samples (100.0%)
✅ GlobalExperimentRunner initialized successfully
📊 Expected total experiments: 165
✅ Eligible target countries: 55

📋 Target Country Details:
   1. USA: 38,379 samples (100.0% valid)
   2. CAN: 5,369 samples (100.0% valid)
   3. NLD: 1,466 samples (100.0% valid)
   4. ISR: 1,622 samples (100.0% valid)
   5. BMU: 757 samples (100.0% valid)
   6. GBR: 10,881 samples (100.0% valid)
   7. JPN: 23,682 samples (100.0% valid)
   8. IRL: 939 samples (100.0% valid)
   9. CHN: 23,057 samples (100.0% valid)
  10. SWE: 3,163 samples (100.0% valid)
  11. 

🌍 Global Experiment Runner initialized. Computing device: CPU
📊 Found 117 countries/regions
📊 Developed countries count: 35
🔍 Analyzing global data quality...


📈 Top 10 Countries by Sample Count:
   1. USA: 38,379 usable samples (100.0%)
   2. JPN: 23,682 usable samples (100.0%)
   3. CHN: 23,057 usable samples (100.0%)
   4. GBR: 10,881 usable samples (100.0%)
   5. KOR: 10,737 usable samples (100.0%)
   6. TWN: 8,959 usable samples (100.0%)
   7. IND: 7,739 usable samples (100.0%)
   8. AUS: 6,644 usable samples (100.0%)
   9. HKG: 6,089 usable samples (100.0%)
  10. CAN: 5,369 usable samples (100.0%)
🎯 Will run experiments on 1 specified countries
📋 Experiment Plan:
   - Target countries: ['AUS']
   - Experiment types: ['A', 'B', 'B_dev']
   - Minimum samples: 50

[ 1/1] Country: AUS
Country data: Total samples 6,644, Valid samples 6,644 (100.0%)
  🏠 Experiment A: AUS Self-Modeling
    📊 Original samples: 6,644, Usable samples: 2,220 (33.4%)
    📊 Experiment A (Country Self-Modeling) - AUS Dataset Split and Sources:
      📍 Data Source Strategy: Use only target country AUS local data
      📈 AUS Total Usable Samples: 2,220
      🏋️ Trainin

...

    📊 Global training samples: 85,110, Target test samples: 113
    📊 Experiment B (Global Baseline) - VNM Dataset Split and Sources:
      📍 Data Source Strategy: Global data training → Target country testing
      🌍 Global Training Data (excluding VNM):
        🏋️ Training Set: 59,576 samples (70.0%) - Source: Multi-country global data (excluding VNM)
        ✅ Validation Set: 12,767 samples (15.0%) - Source: Multi-country global data (excluding VNM)
      🎯 Target Test Data:
        🎯 Test Set: 113 samples - Source: VNM country data
      💡 Note: Train on global data, test cross-domain generalization on target country


    ✅ Completed: R² = 0.3398
  🌍 Experiment B_dev: Developed Countries→VNM Baseline Modeling


    📊 Developed country training samples: 59,096, Target country test samples: 113


    ✅ Completed: R² = -0.1429
  📊 VNM Complete: Average R² = 0.3660 (3/3 experiments successful)
     - LGBM-A-VNM: R² = 0.9010
     - LGBM-B-VNM: R² = 0.3398
     - LGBM-B_dev-VNM: R² = -0.1429

🎉 Global Experiments Complete!
📊 Experiment Statistics:
   - Total experiments: 3
   - Successful experiments: 3
   - Failed experiments: 0
   - Success rate: 100.0%

📈 Successful Experiment Results Summary:
   - Experiment A (Self-Modeling): 1 successful, Average R² = 0.9010
   - Experiment B (Global Baseline): 3 successful, Average R² = 0.3660
   - Experiment B_dev (Developed Countries Baseline): 1 successful, Average R² = -0.1429
✅ Checkpoint saved: 55 countries, 165 results

🎉 CPU experiments complete!
Total time: 1562.26 minutes
Completed experiments: 165
❌ Error during experiment: No module named 'openpyxl'

✅ Experiment execution phase complete!
⚠️ No experiment results, please check data and configuration


Traceback (most recent call last):
  File "/tmp/ipykernel_6957/4134517578.py", line 202, in <module>
    excel_path = save_results_excel(
  File "/tmp/ipykernel_6957/4134517578.py", line 81, in save_results_excel
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
  File "/root/miniconda3/lib/python3.10/site-packages/pandas/io/excel/_openpyxl.py", line 57, in __init__
    from openpyxl.workbook import Workbook
ModuleNotFoundError: No module named 'openpyxl'


In [4]:
# --- Save Results to CSV ---

print("💾 Saving experiment results...")
print("=" * 60)

if 'results_df' in locals() and len(results_df) > 0:
    # Save full results
    output_path = os.path.join(results_dir, 'cpu_ab_experiments_full_results.csv')
    results_df.to_csv(output_path, index=False)
    print(f"✅ Full results saved to: {output_path}")
    
    # Save summary by experiment type
    summary_path = os.path.join(results_dir, 'cpu_ab_experiments_summary.csv')
    summary_by_exp = results_df.groupby('experiment_type').agg({
        'R2_Average': ['count', 'mean', 'std', 'min', 'max'],
        'MAE_Average': 'mean',
        'RMSE_Average': 'mean'
    }).round(4)
    summary_by_exp.to_csv(summary_path)
    print(f"✅ Summary saved to: {summary_path}")
    
    print(f"\n📊 Saved Results:")
    print(f"  - Total experiments: {len(results_df)}")
    print(f"  - Countries: {results_df['country'].nunique()}")
    print(f"  - Experiment types: {list(results_df['experiment_type'].unique())}")
else:
    print("❌ No results to save")

print(f"\n🎉 03a CPU A/B Experiments Complete!")

💾 Saving experiment results...
✅ Full results saved to: /autodl-fs/data/Transfer/results/cpu_ab_experiments_full_results.csv
✅ Summary saved to: /autodl-fs/data/Transfer/results/cpu_ab_experiments_summary.csv

📊 Saved Results:
  - Total experiments: 165
  - Countries: 55
  - Experiment types: ['A', 'B', 'B_dev']

🎉 03a CPU A/B Experiments Complete!
